<a href="https://colab.research.google.com/github/ishanallasanagala/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
#TODO
#Produce revenue column by multiplying quantity and price of each unit
df['revenue'] = df['qty'] * df['price']

#Calculate the total number of units and total revenue
total_units = df['qty'].sum()
total_revenue = df['revenue'].sum()

#Print/report the results
print(f"Total Units: {total_units}")
print(f"Total Revenue: {total_revenue}")
print(f"Interpretation: The 400 randomly generated orders resulted in the production of {total_units} and a total revenue of ${total_revenue}")

Total Units: 783
Total Revenue: 8520.0
Interpretation: The 400 randomly generated orders resulted in the production of 783 and a total revenue of $8520.0


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
# TODO
#Group by category, sum the revenue, and sort in descending order
by_category = (df.groupby('category')['revenue']).sum().sort_values(ascending=False).reset_index()

#Calculate the total share of revenue as a percentage
by_category['share'] = (by_category['revenue'] / by_category['revenue'].sum() * 100).round(1)

#Print interpretation with info about Q2
top_product = by_category.iloc[0]
print(f"Interpretation: {top_product['category']} generated the largest share of stadium sales at ${top_product['revenue']} ({top_product['share']}% of total revenue).")

Interpretation: Food generated the largest share of stadium sales at $4293.0 (50.4% of total revenue).


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# calculating the totals for average revenue and order count per vendor
# Then getting the counts and means
vendor_summary = (df.groupby('vendor_id').agg(
    average_revenue=('revenue', 'mean'),
    order_count=('vendor_id', 'count')
).sort_values('average_revenue', ascending=False)
)
vendor_summary

# Print interpretation sentence# Interpretation
top_vendor = vendor_summary.index[0]
top_avg = vendor_summary.iloc[0]['average_revenue']
top_count = int(vendor_summary.iloc[0]['order_count'])

print(f"\nVendor {top_vendor} had the highest average order revenue at ${top_avg:.2f} across {top_count} orders. ")


Vendor V-01 had the highest average order revenue at $22.60 across 94 orders. 


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# TODO
merch_revenue = df[df['category'] == 'Merch']['revenue'].sum()
total_revenue = df['revenue'].sum()
merch_share = round((merch_revenue / total_revenue) * 100, 1)

print(f"{merch_share}%")

print(f"Merch brought in {merch_share}% of the total revenue brought in.")

20.8%
Merch brought in 20.8% of the total revenue brought in.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
from os.path import join
#Lookup the table mentioned inside of the prompt
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
#left join to merge
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one', indicator = True)

#Validate that the rows and revenue total are unchanged throughout by comparing the lengths and revenues and use.2f to round to two floating point decimal places
rows_match = len(joined) == len(df)
rev_match = round(joined['revenue'].sum(), 2) == round(df['revenue'].sum(), 2)
print(f"Rows unchanged ({len(joined)} == {len(df)}): {rows_match}")
print(f"Revenue unchanged (${joined['revenue'].sum():,.2f} == ${df['revenue'].sum():,.2f}): {rev_match}")

#Find the missing vendor
missing_id = joined[joined['_merge'] == 'left_only']['vendor_id'].unique()[0]
missing_orders = (joined['vendor_id'] == missing_id).sum()
missing_revenue = joined[joined['vendor_id'] == missing_id]['revenue'].sum()

print(f"Unmatched vendor: {missing_id} ({missing_orders} orders, ${missing_revenue:,.2f})")

# 4. Fill in the missing vendor name so that the rows aren't dropped
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown vendor (V-18)')

Rows unchanged (400 == 400): True
Revenue unchanged ($8,520.00 == $8,520.00): True
Unmatched vendor: V-18 (108 orders, $2,349.00)


**The unmatched vendor, and what I did about it:** Vendor V-18 appears in the orders dataset but is absent from vendor_names, accounting for roughly 100 orders and over $2,000 in revenue that is mismatched between the two. Rather than dropping these rows (which would result in a severe inaccuracy in the stadium revenue), I used .fillna('Unknown Vendor (V-18)') to transmit every transaction between both DataFrames and keep the corrent revenue total.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
# TODO
#Build a pivot table with row and column totals for each category
pd.pivot_table(
    joined,
    values='revenue',
    index='vendor_name',
    columns='category',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
).round(2)

print(f"I produced a pivot table showing the sums the revenue for each vendor for each specific category of products")

I produced a pivot table showing the sums the revenue for each vendor for each specific category of products


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

I would tell these vendors to prioritize food prep and staffing over rain gear for next game because food drove 51.0% of total revenue while rain gear accounted for merely 8.5% of of total revenue. I would also tell these vendors to double check that the maintenance and operations teams behind crunching the statistics and data analytics are doing their jobs thoroughly. This would be in the effort of reducing mistakes, such as forgetting to add vendors in certain DataFrames, given that vendor V-18 wasn't registered in vendor_names along with its 99 orders.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

In my opinion, Q5's fix for the missing vendor is the least trustworthy because it presents an inherently reactive solution to fixing a scenario in which a vendor or potentiall ymultiple vendors were missed or mismatched. Although manually running the code in Q5 would instantly join all the vneodrs together, it doesn't contain the solution to who caused the bug to happen or what type of corruption could have been present in one or both DataFrames.